In [ ]:
import pandas as pd
import numpy as np
import math as ms
from itertools import repeat

In [ ]:
sizer_counter = 0

Es = 29000.0 #[ksi]
Fy = 50.0 #[ksi]
f_prime_c = 4 #[ksi]
LB_TO_KG = 0.45359237 #[kg/lb] take-off masses below are in lb; the embodied carbon factors are per kg

def sizer_and_calc(Girder_Span, Infill_Span, Conc_Type_num, Deck_Type, Clear_Spans, 
                   Building_Use_num, floor_length, floor_width, ):

    global sizer_counter
    sizer_counter += 1

    print(f"Run #{sizer_counter}: Girder={Girder_Span}ft, Infill={Infill_Span}ft, Conc={Conc_Type_num}, Deck={Deck_Type}in, Bldg={Building_Use_num}")

    def mitigation_method(Infill_Upsize, Girder_Upsize, Additional_Topping):

        #--------------------------------------------------------------------
        #----------------------Database Imports------------------------------
        #-------------------------------------------------------------------- 

        AISC_database = pd.read_csv('AISC STEEL MANUAL V16 (Modified) Sorted.csv', encoding='latin1')
        Deck_database = pd.read_csv('DeckData.csv')

        #Select only the W-Shapes
        W_Shapes = AISC_database[(AISC_database['Type'] == 'W')] #& (AISC_database['Economic'] == 1)]

        #Deck Data Pulled from 'DeckData.xlsx'
        Deck_Type_Table = Deck_database['Deck Type']
        Conc_Type_Table = Deck_database['Conc Type']
        Total_Depth = Deck_database['Total']
        Topping_Depth = Deck_database['Topping']
        Deck_Gage = Deck_database['Deck Gage']
        Clear_Span_1 = Deck_database['1Span']
        Clear_Span_2 = Deck_database['2Span']
        Clear_Span_3 = Deck_database['3Span']
        Slab_Weight = Deck_database['Concrete + Deck (psf)']
        Allow_Load_4ft = Deck_database['4ft']
        Allow_Load_5ft = Deck_database['5ft']
        Allow_Load_6ft = Deck_database['6ft']
        Allow_Load_7ft = Deck_database['7ft']
        Allow_Load_8ft = Deck_database['8ft']
        Allow_Load_9ft = Deck_database['9ft']
        Allow_Load_10ft = Deck_database['10ft']
        Allow_Load_11ft = Deck_database['11ft']
        Allow_Load_12ft = Deck_database['12ft']
        Allow_Load_13ft = Deck_database['13ft']
        Allow_Load_14ft = Deck_database['14ft']
        Allow_Load_15ft = Deck_database['15ft']
        Allow_Load_16ft = Deck_database['16ft']
        Deck_Design = Deck_database['Design']
        Deck_Weight = Deck_database['Deck Weight (psf)']

        #Steel Member Data Pulled from '"AISC STEEL MANUAL V16 (Modified).csv"
        shape_label = np.array(W_Shapes['AISC_Manual_Label'])
        SW = W_Shapes['W']
        A = np.array(W_Shapes['A'])
        d = np.array(W_Shapes['d']).astype(float)
        k = np.array(W_Shapes['kdet']).astype(float)
        bf_over_2tf = W_Shapes['bf/2tf']
        Sx = W_Shapes['Sx'].astype(float)
        T = W_Shapes['T'].astype(float)
        Jc = W_Shapes['J'].astype(float)
        rts = W_Shapes['rts'].astype(float)
        ho = W_Shapes['ho'].astype(float)
        Z = W_Shapes['Zx [in3]']
        bf = W_Shapes['bf'].astype(float)
        tf = W_Shapes['tf'].astype(float)
        tw = W_Shapes['tw'].astype(float)
        PhiMpx = W_Shapes['PhiMpx']
        PhiBF = W_Shapes['PhiBF']
        Lp = W_Shapes['Lp [ft]']
        Lr = W_Shapes['Lr [ft]']
        Ix = W_Shapes['Ix [in4]']
        PhiVnx = W_Shapes['PhiVnx']
        Economic = W_Shapes['Economic']
        Groups = W_Shapes['Grouping #']
        Ingroup = W_Shapes['# in Grouping']

        #--------------------------------------------------------------------
        #------------------------Deck Sizer----------------------------------
        #--------------------------------------------------------------------

        # Define dictionaries
        span_data_dict = {
            1: Clear_Span_1,
            2: Clear_Span_2,
            3: Clear_Span_3
        }

        load_data_dict = {
            4: Allow_Load_4ft,
            5: Allow_Load_5ft,
            6: Allow_Load_6ft,
            7: Allow_Load_7ft,
            8: Allow_Load_8ft,
            9: Allow_Load_9ft,
            10: Allow_Load_10ft,
            11: Allow_Load_11ft,
            12: Allow_Load_12ft,
            13: Allow_Load_13ft,
            14: Allow_Load_14ft,
            15: Allow_Load_15ft,
            16: Allow_Load_16ft
        }

        Building_Use_dict = {
            1: ('Paper Office', 11, 0.5, 50, 0.03),
            2: ('Electronic Office', 7, 0.5, 50, 0.025),
            3: ('Residence', 6, 0.5, 40, 0.02),
            4: ('Assembly Area', 0, 0.5, 100, 0.02),
            5: ('Shopping Mall', 0, 1.5, 100, 0.02) #Cut Shopping Malls
        }
        Building_Use, live_load_vibrations, tolerance_limit, LL, DR = Building_Use_dict.get(Building_Use_num, ('', '', '', ''))
        
        if Conc_Type_num == 0:
            Conc_Type = 'NW'
            Conc_ECV = 0.133 #Concrete Embodied Carbon Value
            conc_density = 145 #unit-weight of concrete [pcf]
        else:
            Conc_Type = 'LW'
            Conc_ECV = 0.307 #Concrete Embodied Carbon Value
            conc_density = 110 #unit-weight of concrete [pcf]

        #Dictonary used to provide maxium spacing of 22Ga deck with minimal topping for each concrete and deck type
        spacing_lookup = {
            (0, 1.5): 7.67,
            (1, 1.5): 8.25,
            (0, 2): 9.25,
            (1, 2): 10.17,
            (0, 3): 11.08,
            (1, 3): 12.25,
        }

        # Get the value from the dictionary
        max_starting_infill_spacing = spacing_lookup.get((Conc_Type_num, Deck_Type))
        #print("Max Spacing: " + str(max_starting_infill_spacing))

        NumInfill_unrounded = Girder_Span / max_starting_infill_spacing - 1
        #print(NumInfill_unrounded)
    
        NumInfill = max(1, ms.ceil(NumInfill_unrounded))
        #print(NumInfill)       
            
        #print("NumInfill: " + str(NumInfill))

        infill_spacing = Girder_Span / (NumInfill + 1)
        #print(infill_spacing)
        infill_spacing_rounded = ms.ceil(Girder_Span / (NumInfill + 1))

        #This is additional Concrete load as a result of Additional Concrete Topping because the Vulcraft tables factor out concrete self weight of set thicknesses
        if Conc_Type == 'NW':
            Additional_Topping_Load = 145 * (Additional_Topping / 12) #[psf]
        if Conc_Type == 'LW':
            Additional_Topping_Load = 110 * (Additional_Topping / 12) #[psf]
        
        # Superimposed Dead Load [psf]
        SDL = 20 #assumed for all building uses
        Conc_Load = LL + SDL + Additional_Topping_Load

        # Concrete Type Check
        if len(Conc_Type) != len(Conc_Type_Table):
            Conc_Type = list(repeat(Conc_Type, 90))
        Conc_Type_Check = Conc_Type == Conc_Type_Table

        # Deck Type Check
        Deck_Type_Check = Deck_Type == Deck_Type_Table

        # Span Check
        Span_Data = span_data_dict.get(Clear_Spans)
        Span_Check = Span_Data >= infill_spacing

        # Span Check
        if infill_spacing_rounded < 4:
            infill_spacing_rounded = 4
        Load_Data = load_data_dict.get(infill_spacing_rounded)
        if Load_Data is None:
            print("No deck can be sized")
            deck_pass_or_fail = 0  # fails
            return
        elif Load_Data.empty:
            print("Load Data is empty")
            deck_pass_or_fail = 0  # fails
            return
        else:
            deck_pass_or_fail = 1  # passes

        Load_Check = Conc_Load <= Load_Data

        # Set Intersection of all deck checks
        Deck_Check = Conc_Type_Check & Deck_Type_Check & Span_Check & Load_Check

        # Filter Decks based on Deck_Check
        filtered_Decks = Deck_database[Deck_Check]
        
        # Check if filtered_Decks is empty
        if filtered_Decks.empty:
            print("No deck can be sized")
            deck_pass_or_fail = 0 #fails
            return
        else:
            deck_pass_or_fail = 1 #passes
        
        # Select Lightest Deck
        lowest_Slab_Weight = filtered_Decks['Concrete + Deck (psf)'].idxmin()
        deck_pass_or_fail = []
        selected_deck = filtered_Decks.loc[lowest_Slab_Weight, 'Design']
        #print('selected deck is ' + str(selected_deck))

        # New Slab Weight
        selected_deck_slab_SW = filtered_Decks.loc[lowest_Slab_Weight, 'Concrete + Deck (psf)']
        Final_Slab_SW = selected_deck_slab_SW + Additional_Topping_Load

        # New Slab Thickness
        selected_deck_slab_thickness = filtered_Decks.loc[lowest_Slab_Weight, 'Total'] 
        selected_deck_topping_thickness = filtered_Decks.loc[lowest_Slab_Weight, 'Topping'] # Selected Topping Thickness without Addtional [in]
        selected_deck_thickness = filtered_Decks.loc[lowest_Slab_Weight, 'Deck Type'] # Selected Deck Thickness [in]
        selected_deck_SW = filtered_Decks.loc[lowest_Slab_Weight, 'Deck Weight (psf)']
        Final_Slab_Thickness = selected_deck_topping_thickness + Additional_Topping # Total Selected Slab Thickness [in]
        Final_Deck_Slab_Thickness = Final_Slab_Thickness + selected_deck_thickness # Total Selected Floor System Thicknes [in]
        
        #New Dead Load
        DL = Final_Slab_SW + SDL

        #Average Concrete Depth in Flute
        if Deck_Type == 1.5:
            Conc_Depth_in_Flute = 0.532414
        if Deck_Type == 2:
            Conc_Depth_in_Flute = 1
        if Deck_Type == 3:
            Conc_Depth_in_Flute = 1.5
        
        effective_conc_depth = selected_deck_topping_thickness + Conc_Depth_in_Flute + Additional_Topping

        #--------------------------------------------------------------------
        #----------------------Interior Infill-------------------------------
        #--------------------------------------------------------------------

        #----------------------Initial Values--------------------------------

        SW_II = np.array(SW)[:, np.newaxis]  # Shape: (289, 1)
        Ix_II = np.array(Ix)[:, np.newaxis]
        
        # Trib width of Interior Infill
        Trib_II = Girder_Span / (NumInfill + 1)

        # Interior Infill Dead Load w/ Self Weight (psf)
        if NumInfill == 0:
            DL_II = 0.0
        else: 
            DL_II = DL + (SW_II / Trib_II)

        # Interior Infill Dead Load (klf) = W_DL_II
        if NumInfill == 0:
            W_DL_II = 0.0
        else: 
            W_DL_II = (DL_II * Trib_II) / 1000.0

        # Interior Infill Live Load (klf) = W_LL_II
        if NumInfill == 0:
            W_LL_II = 0.0
        else: 
            W_LL_II = (LL * Trib_II) / 1000.0

        # Interior Infill Total Load (klf) = W_TL_II
        W_TL_II = ((DL_II + LL) * Trib_II) / 1000.0

        # Interior Infill Shear Dead Load (kips) = V_DL_II
        V_DL_II = W_DL_II * Infill_Span / 2.0

        # Interior Infill Shear Live Load (kips) = V_LL_II
        V_LL_II = W_LL_II * Infill_Span / 2.0

        # Interior Infill Shear Self Weight
        V_SW_II = (SW_II / Trib_II) * Infill_Span / 2.0 * Trib_II

        # Interior Infill Shear Factored Load (kips * ft ) = V_FL_II
        V_II_LC1 = 1.4 * V_DL_II
        V_II_LC2 = 1.2 * V_DL_II + 1.6 * V_LL_II
        V_FL_II = np.maximum(V_II_LC1, V_II_LC2)

        # Interior Infill Moment Dead Load (kips * ft ) = M_DL_II
        if NumInfill == 0:
            M_DL_II = 0
        if NumInfill > 0:
            M_DL_II = W_DL_II * Infill_Span ** 2 / 8

        # Interior Infill Moment Live Load (kips * ft ) = M_LL_II
        if NumInfill == 0:
            M_LL_II = 0
        if NumInfill > 0:
            M_LL_II = W_LL_II * Infill_Span ** 2 / 8

        # Interior Infill Moment Factored Load (kips * ft ) = M_FL_II
        M_II_LC1 = 1.4 * M_DL_II
        M_II_LC2 = 1.2 * M_DL_II + 1.6 * M_LL_II
        M_FL_II = np.maximum(M_II_LC1, M_II_LC2)

        # Interior Infill Total Load Deflection DEF_TL_II
        if NumInfill == 0:
            DEF_TL_II = 0
        if NumInfill > 0:
            DEF_TL_II = (5*W_TL_II*(Infill_Span**4)*1728)/(384*Es*Ix_II)

        # Interior Infill Live Load Deflection DEF_LL_II
        if NumInfill == 0:
            DEF_LL_II = 0
        if NumInfill > 0:
            DEF_LL_II = (5*W_LL_II*(Infill_Span**4)*1728)/(384*Es*Ix_II)

        #----------------------Filtering Sizes--------------------------------

        PhiMn_II = np.expand_dims(PhiMpx, axis = 1)
        PhiVnx_II = np.expand_dims(PhiVnx, axis = 1)

        # Flexure Check
        Flex_Check_II = PhiMn_II >= M_FL_II

        # Shear Check
        Shear_Sizes_II = PhiVnx_II >= V_FL_II

        # Live Load Deflection Check
        LL_DEF_check_II = np.array(min(1, Infill_Span * 12 / 360) > DEF_LL_II)

        # Total Load Deflection Check
        TL_DEF_check_II = np.array(Infill_Span * 12 / 240 > DEF_TL_II)

        # Economic Size Condition
        Eco = np.array(Economic == 1)

        # Combined Check

        #print("Shape of Flex_Check_II:", Flex_Check_II.shape)
        #print("Shape of Shear_Sizes_II:", Shear_Sizes_II.shape)
        #print("Shape of LL_DEF_check_II:", LL_DEF_check_II.shape)
        #print("Shape of TL_DEF_check_II:", TL_DEF_check_II.shape)
        combined_check_II = (Flex_Check_II & 
                             Shear_Sizes_II & 
                             LL_DEF_check_II & 
                             TL_DEF_check_II)
        #print("Shape of combined_check_II:", combined_check_II.shape)

        V_DL_II_array = np.array(V_DL_II)[:, np.newaxis]
        combined_check_II_array = np.array(combined_check_II)[:, np.newaxis]
        Groups_array = np.array(Groups)[:, np.newaxis]
        Ingroup_array = np.array(Ingroup)[:, np.newaxis]

        infill_sizes = np.expand_dims(shape_label, axis = 1).astype(str)
        infill_depth = np.expand_dims(d, axis = 1)
        infill_area = np.expand_dims(A, axis = 1)
        infill_Ix = np.expand_dims(Ix, axis = 1)
        infill_sw = np.expand_dims(SW, axis = 1)

        #--------------------------------------------------------------------
        #----------------------Interior Girder-------------------------------
        #--------------------------------------------------------------------

        #Self Weight of Interior Inifll    
        SW_IG = np.expand_dims(np.array(SW), axis = (0)) # Shape: (1, 289)

        #Trib width of Interior Girder
        Trib_IG = Infill_Span # Shape: (1, )
        Trib_II = Girder_Span / (NumInfill + 1) # Shape: (1, )

        #Interior Girder Shear Dead Load (kips) = V_DL_IG
        V_DL_IG = ((SW_IG * Girder_Span / 1000) + 2 * V_DL_II * NumInfill) / 2 # Shape: (289, 289)

        #Interior Girder Shear Dead Load (kips) = V_DL_IG
        V_LL_IG = (2 * V_LL_II * NumInfill) / 2 # Shape: (1, )

        #Interior Girder Shear Factored Load (kips * ft ) = V_FL_IG
        V_IG_LC1 = 1.4 * V_DL_IG
        V_IG_LC2 = 1.2 * V_DL_IG + 1.6 * V_LL_IG
        V_FL_IG = np.maximum(V_IG_LC1, V_IG_LC2) # Shape: (289, 289)

        #Interior Girder Moment from SW (kips*ft) = M_SW_IG
        M_SW_IG = np.array((SW_IG / 1000) * Girder_Span ** 2 / 8) # Shape: (1, 289)

        #Total SW of Infills on Interior Girders (kips * ft)
        Total_SW_II = ((SW_II / 1000) * NumInfill * Trib_IG / Girder_Span) # Shape: (289, 1)

        #Interior Girder Moment Dead Load (kips*ft) = M_DL_IG
        if NumInfill == 0:
            M_DL_IG = M_SW_IG 
        if NumInfill == 1:
            M_DL_IG = M_SW_IG + (2 * V_DL_II * Girder_Span / 4)
        if NumInfill == 2:
            M_DL_IG = M_SW_IG + (2 * V_DL_II * Trib_II)
        if NumInfill == 3:
            M_DL_IG = M_SW_IG + (2 * (2 * V_DL_II * Girder_Span / 4))
        if NumInfill > 3:
            M_DL_IG = M_SW_IG + ((DL / 1000) * Trib_IG) * Girder_Span ** 2 / 8 + (Total_SW_II * Girder_Span ** 2 / 8) # Shape: (289 289)

        #Interior Girder Moment Live Load (kips*ft) = M_DL_IG
        if NumInfill == 0:
            M_LL_IG = 0
        if NumInfill == 1:
            M_LL_IG = (2 * V_LL_II * Girder_Span / 4)
        if NumInfill == 2:
            M_LL_IG = (2 * V_LL_II * Trib_II)
        if NumInfill == 3:
            M_LL_IG = (2 * (2 * V_LL_II * Girder_Span / 4))
        if NumInfill > 3:
            M_LL_IG = ((LL / 1000) * Trib_IG) * Girder_Span ** 2 / 8 # Shape: (1, )

        #Interior Girder Moment Factored Load (kips * ft ) = M_FL_IG
        M_IG_LC1 = 1.4 * M_DL_IG
        M_IG_LC2 = 1.2 * M_DL_IG + 1.6 * M_LL_IG
        M_FL_IG = np.maximum(M_IG_LC1, M_IG_LC2) # Shape: (289, 289)

        # Expand Ix to allow for NumPy vectorization
        Ix_array = np.expand_dims(Ix, axis = 0) # Shape: (1, 289)

        #Interior Girder Deflection Self Weight (in) = DEF_SW_IG
        DEF_SW_IG = (5 * (SW_IG / 1000) * (Girder_Span ** 4) * 1728) / (384 * Es * Ix_array) # Shape: (289, 289)

        #Interior Girder Live Load Deflection (in) = DEF_LL_IG
        if NumInfill == 0:
            DEF_LL_IG = 0
        if NumInfill == 1:
            DEF_LL_IG = ((2 * V_LL_II * (Girder_Span ** 3) * 1728) / (48 * Es * Ix_array))
        if NumInfill == 2:
            DEF_LL_IG = ((23 * 2 * V_LL_II * (Girder_Span ** 3) * 1728) / (648 * Es * Ix_array))
        if NumInfill == 3:
            DEF_LL_IG = ((22 * 2 * V_LL_II * (Girder_Span ** 3) * 1728) / (768 * 29000 * Ix_array)) + ((2 * V_LL_II * (Girder_Span ** 3) * 1728) / (48 * 29000 * Ix_array))
        if NumInfill > 3:
            DEF_LL_IG = (5 * ((LL / 1000) * Trib_IG) * (Girder_Span ** 4) * 1728) / (384 * Es * Ix_array) # Shape: (1, )

        #Interior Girder Total Load Deflection (in) = DEF_TL_IG
        if NumInfill == 0:
            DEF_TL_IG = DEF_SW_IG
        if NumInfill == 1:
            DEF_TL_IG = DEF_SW_IG + DEF_LL_IG + ((2 * V_DL_II * (Girder_Span ** 3) * 1728) / (48 * Es * Ix_array))
        if NumInfill == 2:
            DEF_TL_IG = DEF_SW_IG + DEF_LL_IG + ((23 * 2 * V_DL_II * (Girder_Span ** 3) * 1728) / (648 * Es * Ix_array))
        if NumInfill == 3:
            DEF_TL_IG = DEF_SW_IG + DEF_LL_IG + ((22 * 2 * V_DL_II * (Girder_Span ** 3) * 1728) / (768 * Es * Ix_array)) + ((2 * V_DL_II * (Girder_Span ** 3) * 1728) / (48 * 29000 * Ix_array))
        if NumInfill > 3:
            DEF_TL_IG = DEF_SW_IG + DEF_LL_IG + (5 * ((DL / 1000) * Trib_IG) * (Girder_Span ** 4) * 1728) / (384 * Es * Ix_array) + (5 * Total_SW_II * (Girder_Span ** 4) * 1728) / (384 * Es * Ix_array) # Shape: (289, 289)

        #Cb Values (Table 3-1)
        if NumInfill == 0:
            Cb = 1.14
        if NumInfill == 1:
            Cb = 1.67
        if NumInfill == 2:
            Cb = 1
        if NumInfill == 3:
            Cb = 1.11
        if NumInfill > 3:
            Cb = 1

        #----------------------Filtering Sizes--------------------------------

        Lp_expand = np.expand_dims(Lp, axis = 0)
        Lr_expand = np.expand_dims(Lr, axis = 0)
        Z_expand = np.expand_dims(Z, axis = 0)
        PhiMpx_expand = np.expand_dims(PhiMpx, axis = 0)
        PhiBF_expand = np.expand_dims(PhiMpx, axis = 0)
        PhiVnx_expand = np.expand_dims(PhiVnx, axis = 0)
        rts_expand = np.expand_dims(rts, axis = 0)
        Jc_expand = np.expand_dims(Jc, axis = 0)
        Sx_expand = np.expand_dims(Sx, axis = 0)
        ho_expand = np.expand_dims(ho, axis = 0)

        #Flexure Check (Lb<=Lp)
        Lb = Girder_Span / (NumInfill + 1)
        Compact = Lb <= Lp
        Compact_PhiMn = np.where(Compact, PhiMpx, np.nan)

        #Flexure Check (Lp < Lb <=Lr)
        Noncompact1 = Lp < Lb
        Noncompact2 = Lb <= Lr
        Noncompact_LTB = Cb*(PhiMpx-PhiBF*(Lb-Lp))
        Noncompact_PhiMn = np.where(Noncompact1 & Noncompact2, np.minimum(Noncompact_LTB, PhiMpx), np.nan)

        #Flexure Check (Lr < Lb)
        Slender = Lb > Lr
        Fcr = ((Cb*(np.pi)**2*Es)/(Lb*12/rts)**2)*np.sqrt(1+0.078*(Jc/(Sx*ho))*(Lb*12/rts)**2)
        Slender_LTB = 0.9*((Fcr*Sx)/12)
        Slender_PhiMn = np.where(Slender, np.minimum(Slender_LTB, PhiMpx), np.nan)

        PhiMn_IG = np.where(~np.isnan(Compact_PhiMn), Compact_PhiMn, np.where(~np.isnan(Noncompact_PhiMn), Noncompact_PhiMn, Slender_PhiMn))
        PhiVnx_IG = np.expand_dims(PhiVnx, axis = 0)

        #Flexure Check
        Flex_Sizes_IG = M_FL_IG <= PhiMn_IG
        #print("Shape of M_FL_IG:", M_FL_IG.shape)
        #print("Shape of PhiMn_IG:", PhiMn_IG.shape)

        #Shear Check
        Shear_Sizes_IG = V_FL_IG <= PhiVnx_IG

        #Live Load Deflection Check
        LL_DEF_check_IG = DEF_LL_IG <= min(Girder_Span * 12 / 360, 1)

        #Total Load Deflection Check
        TL_DEF_check_IG = DEF_TL_IG <= Girder_Span * 12 / 240

        # Infill fitting inside girders
        infill_d = np.expand_dims(d, axis = 1)
        girder_d = np.expand_dims(d, axis = 0)
        girder_k = np.expand_dims(k, axis = 0)
        depth_check = girder_d - girder_k >= infill_d

        # Combined Check
        #print("Shape of Flex_Sizes_IG:", Flex_Sizes_IG.shape)
        #print("Shape of Shear_Sizes_IG:", Shear_Sizes_IG.shape)
        #print("Shape of LL_DEF_check_IG:", LL_DEF_check_IG.shape)
        #print("Shape of TL_DEF_check_IG:", TL_DEF_check_IG.shape)
        #print("Shape of depth_check:", depth_check.shape)
        #print("Shape of Eco:", Eco.shape)
        combined_check_IG = (Flex_Sizes_IG & 
                             Shear_Sizes_IG & 
                             LL_DEF_check_IG & 
                             TL_DEF_check_IG &
                             depth_check &
                             Eco)
        #print("Shape of combined_check_IG:", combined_check_IG.shape)

        combined_check = combined_check_II & combined_check_IG

        #print("Shape of combined_check:", combined_check.shape)
        #print("Shape of Groups_array:", Groups_array.shape)

        # Create a mask that selects the most economic infill of each grouping
        groups_filtered = pd.DataFrame(np.where(combined_check, Groups_array, np.nan))
        ingroup_filtered = np.where(combined_check, Ingroup_array, np.nan)

        mask = pd.DataFrame(False, index=groups_filtered.index, columns=groups_filtered.columns)

        for column in groups_filtered.columns:
            # Mark the first occurrence of each unique value
            first_occurrences = groups_filtered[column].duplicated(keep='first') == False
    
            # Set the corresponding indices in the mask to True
            mask.loc[first_occurrences, column] = True  # pandas 3 Copy-on-Write safe; identical result (see PATCH_LOG.md)

        final_check = combined_check & mask


        girder_sizes = np.expand_dims(shape_label, axis = 0).astype(str)
        girder_depth = np.expand_dims(d, axis = 0)
        girder_area = np.expand_dims(A, axis = 0)
        girder_Ix = np.expand_dims(Ix, axis = 0)
        girder_sw = np.expand_dims(SW, axis = 0)

        #--------------------------------------------------------------------
        #----------------------Vibration Calc--------------------------------
        #--------------------------------------------------------------------

        dead_load = 4
        damping_ratio = DR

        if Deck_Type == 1.5:
            Conc_Depth_in_Flute = 0.532414
        if Deck_Type == 2:
            Conc_Depth_in_Flute = 1
        if Deck_Type == 3:
            Conc_Depth_in_Flute = 1.5

        effective_conc_depth = selected_deck_topping_thickness + Additional_Topping

        if Conc_Type_num == 0:
            conc_density = 145 #pcf
        if Conc_Type_num == 1:
            conc_density = 110 #pcf

        #Deck Properties
        Ec = conc_density ** 1.5 * ms.sqrt(f_prime_c) #ksi

        #n Calc
        n = Es / (1.35 * Ec)

        #Effective Concrete Slab Width for Infills
        effective_conc_width = min(0.4 * Infill_Span * 12, infill_spacing * 12) #in

        #Transformed Concrete Slab Width for Infills
        transformed_conc_width = effective_conc_width / n #in

        #Transformed Concrete Slab Area for Infills
        transformed_conc_area = effective_conc_depth * transformed_conc_width #in^2

        #Transformed Moment of Inertia for Infills
        y_bar_infill = (transformed_conc_area * (infill_depth / 2 + Deck_Type + effective_conc_depth / 2)) / (transformed_conc_area + infill_area)

        Ij = (transformed_conc_width * effective_conc_depth ** 3 / 12 + transformed_conc_area * 
              (infill_depth / 2 + Deck_Type + effective_conc_depth / 2 - y_bar_infill) ** 2 + infill_Ix + infill_area * y_bar_infill ** 2)

        #Infill uniformly distributed load
        wj = infill_spacing*(live_load_vibrations + Final_Slab_SW + dead_load) + infill_sw

        #Infill deflection using Transformed Moment of Intertia
        DEF_j = (5 * wj * Infill_Span ** 4 * 1728) / (384 * (Es * 1000) * Ij)

        #Infill fundamental frequency
        fj = 0.18 * np.sqrt(386 / DEF_j)

        #transformed slab moment of inertia per unit width
        de = effective_conc_depth + Deck_Type / 2
        Ds = (12 * de ** 3) / (12 * n)

        #The transformed moment of inertia per unit width for Infills
        Dj = Ij / infill_spacing

        #The effective beam panel width of Infills
        Cj = 2.0
        Bj = Cj * (Ds / Dj) ** 0.25 * Infill_Span

        #is Bj <= (2/3) * floor width
        floor_width_check = (2/3) * floor_width
        Bj_check = Bj <= floor_width_check

        #The weight of the beam panel (x1.5 because I am assuimng girder is continuous)
        Wj = 1.5 * (wj / infill_spacing) * Bj * Infill_Span

        #Girder Transformed Moment of Inertia
        #The effective concrete slab width
        effective_conc_width_girder = min(0.2 * Girder_Span, 0.5 * Infill_Span) * 12 + min(0.2 * Girder_Span, 0.5 * Infill_Span) *12 #in

        #Transformed Concrete Slab Width for Girders
        transformed_conc_width_girder = effective_conc_width_girder / n #in

        #Transformed Concrete Slab Width of Deck
        transformed_conc_width_deck = effective_conc_width_girder / 2 / n #in

        #Transformed Concrete Slab Area for Girders
        transformed_conc_area_girder = effective_conc_depth * transformed_conc_width_girder #in^2

        #Transformed Concrete Slab Area of Deck
        transformed_conc_area_deck = Deck_Type * transformed_conc_width_deck #in^2

        # Transformed Moment of Inertia for Girders
        y_bar_girder = (transformed_conc_area_girder * (girder_depth / 2 + Deck_Type + effective_conc_depth / 2) + transformed_conc_area_deck * 
                        (girder_depth / 2 + Deck_Type / 2)) / (transformed_conc_area_girder + transformed_conc_area_deck + girder_area)

        Ig = (transformed_conc_width_girder * effective_conc_depth ** 3 / 12 + transformed_conc_area_girder * (girder_depth / 2 + Deck_Type + effective_conc_depth / 2 - y_bar_girder) ** 2 
              + transformed_conc_width_deck * Deck_Type ** 3 / 12 + transformed_conc_area_deck * (girder_depth / 2 + Deck_Type / 2 - y_bar_girder) ** 2 + girder_Ix + girder_area * y_bar_girder ** 2)
        
                #Girder uniformly distributed load
        wg = Infill_Span * (wj / infill_spacing) + girder_sw #plf

        #Girder deflection using Transformed Moment of Intertia
        DEF_g = (5 * wg * Girder_Span ** 4 *1728) / (384 * 29 * 10 ** 6 * Ig) #in

        #Infill fundamental frequency
        fg = 0.18 * np.sqrt(386 / DEF_g) #Hz

        #The transformed moment of inertia per unit width for Girders
        Dg = Ig / Infill_Span #in^4 / ft

        #The effective beam panel width of Girders
        Cg = 1.8 #Assuming infills are connected to Girder Web
        Bg = Cg * (Dj / Dg) ** 0.25 * Girder_Span #ft

        #is Bg <= (2/3) * floor length
        floor_length_check = (2/3) * floor_length
        Bg_check = Bg <= floor_length_check

        #The weight of the girder panel
        Wg = (wg / Infill_Span) * Bg * Girder_Span #lbs

        #Floor Fundamental Frequency
        fn = 0.18 * np.sqrt(386 / (DEF_j + DEF_g)) #Hz

        #Girder Deflection Reduction
        girder_deflection_reduction = Girder_Span < Bj
        DEF_g_reduced = np.where(girder_deflection_reduction, np.maximum(((Girder_Span / Bj) * DEF_g), 0.5 * DEF_g), DEF_g)

        #equivalent panel mode panel weigh
        W = DEF_j / (DEF_j + DEF_g_reduced) * Wj + DEF_g_reduced / (DEF_j + DEF_g_reduced) * Wg

        #ratio of the peak floor acceleration to the acceleration of gravity
        ap_over_g = (65 * ms.e ** (-0.35 * fn)) / (damping_ratio * W)

        #peak acceleration
        peak_acceleration = ap_over_g * 100
        vibration_check = peak_acceleration <= tolerance_limit

        #--------------------------------------------------------------------
        #---------------------------EC Calcs---------------------------------
        #--------------------------------------------------------------------

        #Bay Area
        bay_area = Infill_Span * Girder_Span

        #Steel Weight
        steel_w_infill = infill_sw * Infill_Span * (NumInfill + 1) #interior infills + one edge infill; the bay is a repeating unit
        steel_w_girder = girder_sw * Girder_Span * 1 #one of the two girders; the bay is a repeating unit
        steel_w = steel_w_infill + steel_w_girder #lbs

        #Steel deck weight
        steel_deck_w = selected_deck_SW * bay_area

        #Conc Weight
        conc_w = (selected_deck_topping_thickness + Conc_Depth_in_Flute + Additional_Topping) / 12 * conc_density * bay_area #lbs

        #Embodied Carbon
        EC = (steel_w * 1.22 + steel_deck_w * 2.32 + conc_w * Conc_ECV) * LB_TO_KG #kgCO2e

        #Embodied Carbon Scaled
        EC_scaled = EC / bay_area #kgCO2e/ft^2
        

        #Dictionary of variables to be used outside defenition
        results = {
            'DR': damping_ratio,
            'Peak Acceleration': peak_acceleration,
            'Embodied Carbon Scaled': EC_scaled,
            'Infill Sizes': infill_sizes,
            'Girder Sizes': girder_sizes,
            'Groups Array': Groups_array,
            'Groups': Groups,
            'Final Check': final_check,
            'M_FL_II': M_FL_II,
            'M_FL_IG': M_FL_IG,
            'PhiMn_II': PhiMpx,
            'DEF_LL_II': DEF_LL_II,
            'DEF_LL_IG': DEF_LL_IG,
            'DEF_TL_II': DEF_TL_II,
            'DEF_TL_IG': DEF_TL_IG,
            'effective': effective_conc_depth,
            'Total Topping': Final_Slab_Thickness,
            'Fundamental Frequency': fn,
            'DL_II': DL_II,

            # Initial variables and deck calculations
            'SDL': SDL,
            'Conc_Load': Conc_Load,
            'Conc_Type': Conc_Type,
            'Deck_Type': Deck_Type,
            'Span_Data': Span_Data,
            'Load_Data': Load_Data,
            'deck_pass_or_fail': deck_pass_or_fail,
            'selected_deck': selected_deck,
            'selected_deck_slab_SW': selected_deck_slab_SW,
            'Final_Slab_SW': Final_Slab_SW,
            'selected_deck_slab_thickness': selected_deck_slab_thickness,
            'selected_deck_topping_thickness': selected_deck_topping_thickness,
            'selected_deck_thickness': selected_deck_thickness,
            'selected_deck_SW': selected_deck_SW,
            'Final_Slab_Thickness': Final_Slab_Thickness,
            'Final_Deck_Slab_Thickness': Final_Deck_Slab_Thickness,
            'DL': DL,
            'Conc_Depth_in_Flute': Conc_Depth_in_Flute,
            'effective_conc_depth': effective_conc_depth,

            # Interior Infill tracking
            'SW_II': SW_II,
            'Ix_II': Ix_II,
            'Trib_II': Trib_II,
            'DL_II': DL_II,
            'W_DL_II': W_DL_II,
            'W_LL_II': W_LL_II,
            'W_TL_II': W_TL_II,
            'V_DL_II': V_DL_II,
            'V_LL_II': V_LL_II,
            'V_SW_II': V_SW_II,
            'V_FL_II': V_FL_II,
            'M_DL_II': M_DL_II,
            'M_LL_II': M_LL_II,
            'M_FL_II': M_FL_II,
            'DEF_TL_II': DEF_TL_II,
            'DEF_LL_II': DEF_LL_II,

            # Filtering sizes tracking
            'Flex_Check_II': Flex_Check_II,
            'Shear_Sizes_II': Shear_Sizes_II,
            'LL_DEF_check_II': LL_DEF_check_II,
            'TL_DEF_check_II': TL_DEF_check_II,
            'Eco': Eco,
            'combined_check_II': combined_check_II,
            'V_DL_II_array': V_DL_II_array,
            'combined_check_II_array': combined_check_II_array,
            'Groups_array': Groups_array,
            'Ingroup_array': Ingroup_array,
            'infill_sizes': infill_sizes,
            'infill_depth': infill_depth,
            'infill_area': infill_area,
            'infill_Ix': infill_Ix,
            'infill_sw': infill_sw,
            'infill_d': infill_d,

            # Interior Girder tracking
            'SW_IG': SW_IG,
            'Trib_IG': Trib_IG,
            'V_DL_IG': V_DL_IG,
            'V_LL_IG': V_LL_IG,
            'V_FL_IG': V_FL_IG,
            'M_SW_IG': M_SW_IG,
            'Total_SW_II': Total_SW_II,
            'M_DL_IG': M_DL_IG,
            'M_LL_IG': M_LL_IG,
            'M_FL_IG': M_FL_IG,
            'DEF_SW_IG': DEF_SW_IG,
            'DEF_LL_IG': DEF_LL_IG,
            'DEF_TL_IG': DEF_TL_IG,
            'Cb': Cb,

            # Filtering sizes expanded variables tracking
            'Lp_IG': Lp_expand,
            'Lr_IG': Lr_expand,
            'Z_expand': Z_expand,
            'PhiMpx_expand': PhiMpx_expand,
            'PhiBF_expand': PhiBF_expand,
            'PhiVnx_expand': PhiVnx_expand,
            'rts_expand': rts_expand,
            'Jc_expand': Jc_expand,
            'Sx_expand': Sx_expand,
            'ho_expand': ho_expand,
            'Lb': Lb,
            'Compact': Compact,
            'Compact_PhiMn': Compact_PhiMn,
            'Noncompact1': Noncompact1,
            'Noncompact2': Noncompact2,
            'Noncompact_LTB': Noncompact_LTB,
            'Noncompact_PhiMn': Noncompact_PhiMn,
            'Slender': Slender,
            'Fcr': Fcr,
            'Slender_LTB': Slender_LTB,
            'Slender_PhiMn': Slender_PhiMn,
            'PhiMn_II': PhiMn_II,
            'PhiVnx_II': PhiVnx_II,
            'PhiMn_IG': PhiMn_IG,
            'PhiVnx_IG': PhiVnx_IG,

            # Combined checks tracking
            'Flex_Sizes_IG': Flex_Sizes_IG,
            'Shear_Sizes_IG': Shear_Sizes_IG,
            'LL_DEF_check_IG': LL_DEF_check_IG,
            'TL_DEF_check_IG': TL_DEF_check_IG,
            'depth_check': depth_check,
            'combined_check_IG': combined_check_IG,
            'combined_check': combined_check,
            'groups_filtered': groups_filtered,
            'ingroup_filtered': ingroup_filtered,
            'mask': mask,
            'final_check': final_check,
            'girder_sizes': girder_sizes,
            'girder_depth': girder_depth,
            'girder_area': girder_area,
            'girder_Ix': girder_Ix,
            'girder_sw': girder_sw,

            # Vibration calculations tracking
            'damping_ratio': damping_ratio,
            'Ec': Ec,
            'n': n,
            'effective_conc_width': effective_conc_width,
            'transformed_conc_width': transformed_conc_width,
            'transformed_conc_area': transformed_conc_area,
            'y_bar_infill': y_bar_infill,
            'Ij': Ij,
            'wj': wj,
            'DEF_j': DEF_j,
            'fj': fj,
            'Ds': Ds,
            'Dj': Dj,
            'Bj': Bj,
            'floor_width_check': floor_width_check,
            'Bj_check': Bj_check,
            'Wj': Wj,
            'effective_conc_width_girder': effective_conc_width_girder,
            'transformed_conc_width_girder': transformed_conc_width_girder,
            'transformed_conc_width_deck': transformed_conc_width_deck,
            'transformed_conc_area_girder': transformed_conc_area_girder,
            'transformed_conc_area_deck': transformed_conc_area_deck,
            'y_bar_girder': y_bar_girder,
            'Ig': Ig,
            'wg': wg,
            'DEF_g': DEF_g,
            'fg': fg,
            'Dg': Dg,
            'Bg': Bg,
            'floor_length_check': floor_length_check,
            'Bg_check': Bg_check,
            'Wg': Wg,
            'fn': fn,
            'girder_deflection_reduction': girder_deflection_reduction,
            'DEF_g_reduced': DEF_g_reduced,
            'W': W,
            'ap_over_g': ap_over_g,
            'peak_acceleration': peak_acceleration,
            'vibration_check': vibration_check,

            # EC calculations tracking
            'bay_area': bay_area,
            'steel_w_infill': steel_w_infill,
            'steel_w_girder': steel_w_girder,
            'steel_w': steel_w,
            'steel_deck_w': steel_deck_w,
            'conc_w': conc_w,
            'EC': EC,
            'EC_scaled': EC_scaled,
            'girder_d': girder_d,
            'girder_k': girder_k
        }

        return results

    #------------------------------------------------------------------------
    #----------------------Dataframe Creation--------------------------------
    #------------------------------------------------------------------------
    
    # Base calculation
    base_results = mitigation_method(0, 0, 0)

    if base_results == None:
        return None
    
    else:
        combined_matrix = np.vectorize(lambda x, y: f"Infill: {x}, Girder: {y}")(base_results['Infill Sizes'], base_results['Girder Sizes'])
        group_matrix = np.vectorize(lambda x, y: f"Infill: {x}, Girder: {y}")(base_results['Groups Array'], base_results['Groups'])

    #Transforms 1d array to apply to infill values
    def expand_infill(value):
        return np.repeat(np.expand_dims(value, axis=1), repeats=273, axis=1)
    
    #Transforms 1d array to apply to girder values
    def expand_girder(value):
        value_reshape = value.reshape(1, 273)
        return np.repeat(value_reshape, 273, axis=0)

    def map_to_matrices(range_traversal_df, target_matrices_dict):
        """
        Map column and row indices from range_traversal_df to multiple target matrices.

        Parameters:
        range_traversal_df (pd.DataFrame): DataFrame containing ColIdx, RowIdx, ColumnName, and Value.
        target_matrices_dict (dict): Dictionary where keys are matrix names and values are target matrices (numpy arrays).

        Returns:
        pd.DataFrame: DataFrame with the mapped values from all target matrices as new columns.
        """
        mapped_results = range_traversal_df.copy()

        # Iterate over each target matrix in the dictionary
        for matrix_name, target_matrix in target_matrices_dict.items():
            mapped_values = []

            # Map values from the target matrix using the column and row indices
            for _, row in range_traversal_df.iterrows():
                col_idx = row["ColIdx"]
                row_idx = row["RowIdx"]

                # Retrieve the value from the matrix
                try:
                    mapped_value = target_matrix[row_idx, col_idx]
                except IndexError:
                    mapped_value = np.nan  # Handle out-of-bounds indices gracefully

                # Add the mapped value to the list
                mapped_values.append(mapped_value)

            # Add the mapped values as a new column to the DataFrame
            mapped_results[f"{matrix_name}"] = mapped_values

        return mapped_results
    
    #Dictonary of all values returned in final upsizing dataframe
    target_base_matrices = {
        "Damping Ratio": np.full((273, 273),base_results['DR']),
        "Additional Topping": np.zeros_like(base_results['Peak Acceleration']),
        "Peak Acceleration": base_results['Peak Acceleration'],
        "Embodied Carbon Scaled": base_results['Embodied Carbon Scaled'],
        "Combined Sizes": combined_matrix,
        "Infill Sizes": expand_infill(base_results['Infill Sizes'].flatten()),
        "Infill Shear": expand_infill(base_results['V_FL_II']),
        "Infill Shear Capacity": expand_infill(base_results['PhiVnx_II']),
        "Infill Moment": expand_infill(base_results['M_FL_II']),
        "Infill Moment Capcity": expand_infill(base_results['PhiMn_II']),
        "Infill Live Load Deflection": expand_infill(base_results['DEF_LL_II']),
        "Infill Total Load Deflection": expand_infill(base_results['DEF_TL_II']),
        "Girder Sizes": expand_girder(base_results['Girder Sizes'].flatten()),
        "Girder Shear": base_results['V_FL_IG'],
        "Girder Shear Capacity": expand_girder(base_results['PhiVnx_IG']),
        "Girder Moment": base_results['M_FL_IG'],
        "Girder Moment Capcity": expand_girder(base_results['PhiMn_IG']),
        "Girder Live Load Deflection": expand_girder(base_results['DEF_LL_IG']),
        "Girder Total Load Deflection": base_results['DEF_TL_IG'],
        "Total Topping": base_results['Total Topping'],
        "Fundamental Frequency": base_results['Fundamental Frequency']

    }

    def initial_values(base_results):

        #DataFrame of the EC Values
        EC_dataframe = pd.DataFrame(np.where(base_results['Final Check'], base_results['Embodied Carbon Scaled'], np.nan),
                                     index=base_results['Infill Sizes'].flatten(),
                                       columns=base_results['Girder Sizes'].flatten())

        # Find the minimum value and its location
        min_location = EC_dataframe.stack().idxmin()

        # Convert row and column labels to integer indices
        row_index = EC_dataframe.index.get_loc(min_location[0])  # Convert row label to index
        col_index = EC_dataframe.columns.get_loc(min_location[1])  # Convert column label to index

        # Or alternatively, using lists:
        initial_df = pd.DataFrame({
            'ColIdx': [col_index],
            'RowIdx': [row_index]
        })

        # Map the matrices using the updated function
        mapped_results_df = map_to_matrices(initial_df, target_base_matrices)

        return mapped_results_df
    
    #Call Initial Sizing DF
    initial_values_df = initial_values(base_results)

    #-----------------------Infill Upsizing----------------------------------

    def infill_upsizing(base_results):
    
        if initial_values_df is None or initial_values_df['Peak Acceleration'].iloc[0] < 0.5:
            dummy_df = pd.DataFrame(0, index=range(1), columns=list(target_base_matrices.keys()) + ['# of Upsizes'])
            return dummy_df

        #DataFrame of the EC Values
        EC_dataframe = pd.DataFrame(np.where(base_results['Final Check'], base_results['Embodied Carbon Scaled'], np.nan),
                                     index=base_results['Infill Sizes'].flatten(),
                                       columns=base_results['Girder Sizes'].flatten())
        
        # Find the minimum value and its location
        min_location = EC_dataframe.stack().idxmin()

        # Convert row and column labels to integer indices
        row_index = EC_dataframe.index.get_loc(min_location[0])  # Convert row label to index
        col_index = EC_dataframe.columns.get_loc(min_location[1])  # Convert column label to index
        #print(f"Row index: {row_index}, Column index: {col_index}")

        # Create a DataFrame of the PhiMn Grouping
        Grouping_dataframe = base_results['Groups Array']
        
        #DataFrame of the PhiMn Grouping, with final check masking
        Grouping_df = pd.DataFrame(
            np.where(base_results['Final Check'], Grouping_dataframe, np.nan),
            index=base_results['Infill Sizes'].flatten(),
            columns=base_results['Girder Sizes'].flatten()
        )

        #Removes all columns to the right of the lowest observed EC value in matrix to begin Infill Upsizing
        Grouping_df= Grouping_df.iloc[:, :col_index+1]

        # Compute max and min values
        max_values = Grouping_df.max(skipna=True)
        min_values = Grouping_df.min(skipna=True)

        # Convert dictionary to list of tuples for processing
        min_values_list = list(min_values.items())

        # Initialize variables for filtering
        filtered_indices = []
        current_min = float('inf')  # Start with infinity to find the minimum

        # Traverse the list from bottom to top
        for index, item in enumerate(reversed(min_values_list)):
            column_name, value = item
            if value < current_min:
                # Only add the item if it's less than the current minimum seen
                current_min = value
                # Find the row index of this value in the respective column
                row_index = Grouping_df[column_name].tolist().index(value)
                # Add the column index, row index, column name, and value to the filtered_indices
                column_index = list(Grouping_df.columns).index(column_name)
                filtered_indices.append((len(min_values_list) - 1 - index, column_index, row_index, column_name, value))

        # Reverse the filtered_indices to restore the original order
        filtered_indices.reverse()

        # Handle the bottom column index
        bottom_column_index = filtered_indices[-1][0]
        max_value_bottom_column = max_values[bottom_column_index]

        # Create a DataFrame for sorting results
        sorting_df = pd.DataFrame(filtered_indices, columns=["Idx", "ColIdx", "RowIdx", "ColumnName", "Value"])


        # Add the bottom column max value to the sorted group
        sorted_groups = list(sorting_df["Value"]) + [max_value_bottom_column]

        # Initialize an empty list to store ranges
        ranges = []

        # Loop through the sorted_groups to create ranges
        for i in range(len(sorted_groups) - 1):
            start = int(sorted_groups[i])  # Start value of the range
            end = int(sorted_groups[i + 1])  # End value of the range
            ranges.append(list(range(start, end + 1)))  # Add range to the list

        sorting_df["Ranges"] = ranges

        # Now integrate the traverse_ranges function to process ranges
        def traverse_ranges(dataframe, sorting_df):
            """
            Traverse the dataframe for all the range values for each column specified in sorting_df.
            """
            result = []
            
            # Loop through each row in sorting_df to process its ranges
            for _, row in sorting_df.iterrows():
                col_name = row["ColumnName"]
                col_idx = row["ColIdx"]
                col_ranges = row["Ranges"]
                
                # Traverse each value in the range and find its index in the column
                for value in col_ranges:
                    if value in dataframe[col_name].values:
                        row_idx = dataframe[col_name].tolist().index(value)
                        result.append((col_idx, row_idx, col_name, value))
                    else:
                        # If value isn't present, skip it
                        continue
            
            # Convert the result into a DataFrame for easier viewing and processing
            range_traversal_df = pd.DataFrame(result, columns=["ColIdx", "RowIdx", "ColumnName", "Value"])
            return range_traversal_df

        # Call the traverse_ranges function to process ranges
        range_traversal_df = traverse_ranges(Grouping_df, sorting_df)

        # Map the matrices using the updated function
        mapped_results_df = map_to_matrices(range_traversal_df, target_base_matrices)
    
        # Reverse the rows of the DataFrame
        mapped_results_df = mapped_results_df[::-1].reset_index(drop=True)  # Reverse Rows

        # Always add the '# of Upsizes' column, regardless of peak acceleration
        mapped_results_df.insert(0, '# of Upsizes', mapped_results_df.index)
        
        # Check if there are any rows where 'Peak Acceleration' is less than 0.5
        filtered_df = mapped_results_df[mapped_results_df['Peak Acceleration'] < 0.5]

        if filtered_df.empty:
            # If no values less than 0.5, return the full dataset
            print("No 'Peak Acceleration' values found below 0.5.")
        else:
            # Find the index where 'Peak Acceleration' first drops below 0.5
            stop_index = filtered_df.index[0]
            # Slice dataframe once PA dips under 0.5
            mapped_results_df = mapped_results_df.loc[:stop_index]

        return mapped_results_df

    #-----------------------Girder Upsizing----------------------------------

    def girder_upsizing(base_results):

        if initial_values_df is None or initial_values_df['Peak Acceleration'].iloc[0] < 0.5:
            dummy_df = pd.DataFrame(0, index=range(1), columns=list(target_base_matrices.keys()) + ['# of Upsizes'])
            return dummy_df
    
        #DataFrame of the EC Values
        EC_dataframe = pd.DataFrame(np.where(base_results['Final Check'], base_results['Embodied Carbon Scaled'], np.nan),
                                    index=base_results['Infill Sizes'].flatten(),
                                    columns=base_results['Girder Sizes'].flatten())

        # Find the minimum value and its location
        min_location = EC_dataframe.stack().idxmin()

        # Convert row and column labels to integer indices
        row_index = EC_dataframe.index.get_loc(min_location[0])  # Convert row label to index
        col_index = EC_dataframe.columns.get_loc(min_location[1])  # Convert column label to index

        # Create a DataFrame of the PhiMn Grouping
        Grouping_dataframe = base_results['Groups Array']

        #DataFrame of the PhiMn Grouping, with final check masking
        Grouping_df = pd.DataFrame(
            np.where(base_results['Final Check'], Grouping_dataframe, np.nan),
            index=base_results['Infill Sizes'].flatten(),
            columns=base_results['Girder Sizes'].flatten()
        )

        #Removes all columns to the right of the lowest observed EC value in matrix to begin Infill Upsizing
        Grouping_df = Grouping_df.iloc[:, :col_index+1]

        # Initialize a list to store the results
        max_indices = []

        # Iterate over each column to find the max value and its corresponding row
        for col_idx, column_name in enumerate(Grouping_df.columns):
            # Get the column data (drop NaN values)
            column_data = Grouping_df[column_name].dropna()
            
            if not column_data.empty:
                # Get max value and row index for this column
                max_value = column_data.max()
                row_label = column_data.idxmax()  # Row label corresponding to the max value
                row_idx = Grouping_df.index.get_loc(row_label)  # Convert label to numeric index
                max_indices.append((col_idx, row_idx, column_name, max_value))

        # Convert the list to a DataFrame
        max_indicies_df = pd.DataFrame(max_indices, columns=['ColIdx', 'RowIdx', 'ColumnName', 'MaxValue'])
        
        # Map the matrices using the updated function
        mapped_results_df = map_to_matrices(max_indicies_df, target_base_matrices)

        # Reverse the rows of the DataFrame
        mapped_results_df = mapped_results_df[::-1].reset_index(drop=True)  # Reverse Rows

        # Always add the '# of Upsizes' column, regardless of peak acceleration
        mapped_results_df.insert(0, '# of Upsizes', mapped_results_df.index)

        # Check if there are any rows where 'Peak Acceleration' is less than 0.5
        filtered_df = mapped_results_df[mapped_results_df['Peak Acceleration'] < 0.5]

        if filtered_df.empty:
            # If no values less than 0.5, return the full dataset
            print("No 'Peak Acceleration' values found below 0.5.")
        else:
            # Find the index where 'Peak Acceleration' first drops below 0.5
            stop_index = filtered_df.index[0]
            # Slice dataframe once PA dips under 0.5
            mapped_results_df = mapped_results_df.loc[:stop_index]
        
        return mapped_results_df

    #-----------------------Additional Concrete------------------------------

    def concrete_upsizing(cap, step):

        if initial_values_df is None or initial_values_df['Peak Acceleration'].iloc[0] < 0.5:
            dummy_df = pd.DataFrame(0, index=range(1), columns=list(target_base_matrices.keys()) + ['# of Upsizes'])
            return dummy_df
        
        if initial_values_df['Peak Acceleration'].iloc[0] <= 0.5:
            return pd.DataFrame(0, index=range(1), columns=target_base_matrices.keys())

        # Initialize a dictionary to store the results
        all_mitigated_results = {}
        
        # Initialize a list to store peak accelerations for each iteration
        peak_accelerations = []
    
        # Initialize an empty DataFrame to collect results
        all_results_df = pd.DataFrame()

        # Check Additional_Topping
        Additional_Topping = 0

        while Additional_Topping < cap:  # Cap in inches (5 inches = 100 iterations with step 0.1)

            Additional_Topping += step  # Increment in inches
            mitigated_results = mitigation_method(0, 0, Additional_Topping - step)

            if mitigated_results is None:
                break    

            # Store the mitigated results in the dictionary with the key being the Additional_Topping value
            all_mitigated_results[Additional_Topping] = mitigated_results

            EC_dataframe = pd.DataFrame(np.where(mitigated_results['Final Check'], mitigated_results['Embodied Carbon Scaled'], np.nan),
                                         index=base_results['Infill Sizes'].flatten(),
                                           columns=base_results['Girder Sizes'].flatten())

            # Check if EC_dataframe contains only NaN values
            if EC_dataframe.isnull().all().all():
                if all_results_df.isnull().all().all():
                    print("EC_dataframe contains only NaN values. Returning nothing.")
                    return None  # Exit gracefully and return nothing
                else:
                    return all_results_df
            
            # Find the minimum value and its location
            min_location = EC_dataframe.stack().idxmin()

            # Convert row and column labels to integer indices
            row_index = EC_dataframe.index.get_loc(min_location[0])  # Convert row label to index
            col_index = EC_dataframe.columns.get_loc(min_location[1])  # Convert column label to index

            combined_mitigated_matrix = np.vectorize(lambda x, y: f"Infill: {x}, Girder: {y}")(mitigated_results['Infill Sizes'], mitigated_results['Girder Sizes'])

            target_mitigated_matrices = {
            "Damping Ratio": np.full((273, 273),base_results['DR'])[row_index, col_index],
            "Additional Topping": Additional_Topping - step,
            "Peak Acceleration": mitigated_results['Peak Acceleration'][row_index, col_index],
            "Embodied Carbon Scaled": mitigated_results['Embodied Carbon Scaled'][row_index, col_index],
            "Combined Sizes": combined_mitigated_matrix[row_index, col_index],
            "Infill Sizes": expand_infill(mitigated_results['Infill Sizes'].flatten())[row_index, col_index],
            "Infill Moment": expand_infill(mitigated_results['M_FL_II'])[row_index, col_index],
            "Infill Moment Capcity": expand_infill(mitigated_results['PhiMn_II'])[row_index, col_index],
            "Infill Composite Live Load Deflection": expand_infill(mitigated_results['DEF_LL_II'])[row_index, col_index],
            "Infill Composite Total Load Deflection": expand_infill(mitigated_results['DEF_TL_II'])[row_index, col_index],
            "Girder Sizes": expand_girder(mitigated_results['Girder Sizes'].flatten())[row_index, col_index],
            "Girder Moment": mitigated_results['M_FL_IG'][row_index, col_index],
            "Girder Moment Capcity": expand_girder(mitigated_results['PhiMn_IG'])[row_index, col_index],
            "Girder Live Load Deflection": expand_girder(mitigated_results['DEF_LL_IG'])[row_index, col_index],
            "Girder Total Load Deflection": mitigated_results['DEF_TL_IG'][row_index, col_index],
            "Total Topping": mitigated_results["Total Topping"],
            "Fundamental Frequency": mitigated_results['Fundamental Frequency'][row_index, col_index]
            }

            # Use row_index and col_index as a mask to pull the value for peak acceleration or any other value
            peak_acceleration = target_mitigated_matrices['Peak Acceleration']
            #print(f"Peak Acceleration for Additional_Topping = {Additional_Topping - step}: {peak_acceleration}")

            # Convert the target mitigated matrices dictionary to a DataFrame for this iteration
            temp_df = pd.DataFrame([target_mitigated_matrices])

            # Append the current iteration's results to the all_results_df (stacking vertically)
            all_results_df = pd.concat([all_results_df, temp_df], ignore_index=True)

            if peak_acceleration < 0.5:
                break
            
        return(all_results_df)

    def steel_upsizing(base_results):

        # Mask: Only keep EC values where Final Check is True AND Peak Acceleration ≤ 0.5
        mask = np.logical_and(base_results['Final Check'], base_results['Peak Acceleration'] <= 0.5)

        # Apply mask to Embodied Carbon
        masked_ec = np.where(mask, base_results['Embodied Carbon Scaled'], np.nan)

        # Create DataFrame with appropriate index/columns
        EC_dataframe = pd.DataFrame(
            masked_ec,
            index=base_results['Infill Sizes'].flatten(),
            columns=base_results['Girder Sizes'].flatten()
        )

        # Check if EC_dataframe contains only NaN values
        if EC_dataframe.isnull().all().all():
            print(f"No valid sizes found for {percent_comp}% composite action.")
            dummy_df = pd.DataFrame({
                'ColIdx': [0],
                'RowIdx': [0]
            })
            mapped_results = map_to_matrices(dummy_df, target_base_matrices)
            # Set all values to 0
            for col in mapped_results.columns:
                mapped_results[col] = 0
            return mapped_results

        # Find the minimum value and its location
        min_location = EC_dataframe.stack().idxmin()

        # Convert row and column labels to integer indices
        row_index = EC_dataframe.index.get_loc(min_location[0])  # Convert row label to index
        col_index = EC_dataframe.columns.get_loc(min_location[1])  # Convert column label to index

        # Or alternatively, using lists:
        initial_df = pd.DataFrame({
            'ColIdx': [col_index],
            'RowIdx': [row_index]
        })


        # Map the matrices using the updated function
        mapped_results_df = map_to_matrices(initial_df, target_base_matrices)

        return mapped_results_df

    # Call Upsziing Defenitions
    infill_upsizing_df = infill_upsizing(base_results)
    girder_upsizing_df = girder_upsizing(base_results)
    concrete_upsizing_df = concrete_upsizing(10, 0.25)
    steel_upsizing_df = steel_upsizing(base_results)

    initial_ec = initial_values_df['Embodied Carbon Scaled'][0]
    infill_ec = infill_upsizing_df['Embodied Carbon Scaled'].iloc[-1]
    girder_ec = girder_upsizing_df['Embodied Carbon Scaled'].iloc[-1]
    concrete_ec = concrete_upsizing_df['Embodied Carbon Scaled'].iloc[-1]
    steel_ec = steel_upsizing_df['Embodied Carbon Scaled'][0]

    initial_pa = initial_values_df['Peak Acceleration'][0]
    infill_pa = infill_upsizing_df['Peak Acceleration'].iloc[-1]
    girder_pa = girder_upsizing_df['Peak Acceleration'].iloc[-1]
    concrete_pa = concrete_upsizing_df['Peak Acceleration'].iloc[-1]
    steel_pa = steel_upsizing_df['Peak Acceleration'][0]

    if infill_pa > 0.5:
        infill_ec = float('inf')
    if girder_pa > 0.5:
        girder_ec = float('inf')
    if concrete_pa > 0.5:
        concrete_ec = float('inf')
    if steel_pa > 0.5:
        steel_ec = float('inf')


    if (initial_ec != 0) and (initial_pa <= 0.5):
        condition_class = 1
        best_method_df = initial_values_df

    elif (infill_ec != initial_ec and infill_ec != 0 and infill_ec <= min(girder_ec, concrete_ec, steel_ec)) and (infill_pa <= 0.5):
        condition_class = 2
        best_method_df = infill_upsizing_df

    elif (girder_ec != initial_ec and girder_ec != 0 and girder_ec <= min(infill_ec, concrete_ec, steel_ec)) and (girder_pa <= 0.5):
        condition_class = 3
        best_method_df = girder_upsizing_df

    elif (concrete_ec != initial_ec and concrete_ec != 0 and concrete_ec <= min(infill_ec, girder_ec, steel_ec)) and (concrete_pa <= 0.5):
        condition_class = 4
        best_method_df = concrete_upsizing_df
        
    elif (steel_ec != initial_ec and steel_ec != 0 and steel_ec <= min(infill_ec, girder_ec, concrete_ec)) and (steel_pa <= 0.5):
        condition_class = 6
        best_method_df = steel_upsizing_df

    else:
        condition_class = 5
        best_method_df = initial_values_df



    return initial_values_df, infill_upsizing_df, girder_upsizing_df, concrete_upsizing_df, steel_upsizing_df, best_method_df, condition_class

### Single Design Testing

In [ ]:
single_results = sizer_and_calc(Girder_Span=30, Infill_Span=30, Conc_Type_num=1, Deck_Type=2, 
                                Clear_Spans=3, Building_Use_num=3, floor_length=200, floor_width=200)

# Reset counter before starting new sampling run
sizer_counter = 0

single_data = []

if single_results is None:
    print("Error: sizer_and_calc returned None. Check input parameters.")
    single_data.append({
        'Initial Infill Size': 0,
        'Initial Girder Size': 0,
        'Initial Topping': 0,
        'Initial EC_Scaled': 0,
        'Initial Peak Acceleration': 0,
        'Initial Fundamental Frequency': 0,
        'Mitigated Infill # of Upsizes': 0,
        'Mitigated Infill Size': 0,
        'Mitigated Infill Girder Size': 0,
        'Mitigated Infill EC_Scaled': 0,
        'Mitigated Infill Peak Acceleration': 0,
        'Mitigated Infill Fundamental Frequency': 0,
        'Mitigated Girder # of Upsizes': 0,
        'Mitigated Girder Size': 0,
        'Mitigated Girder Infill Size': 0,
        'Mitigated Girder EC_Scaled': 0,
        'Mitigated Girder Peak Acceleration': 0,
        'Mitigated Girder Fundamental Frequency': 0,
        'Mitigated Conc Additional Topping (in)': 0,
        'Mitigated Conc Infill Size': 0,
        'Mitigated Conc Girder Size': 0,
        'Mitigated Conc Topping': 0,
        'Mitigated Conc EC_Scaled': 0,
        'Mitigated Conc Peak Acceleration': 0,
        'Mitigated Conc Fundamental Frequency': 0,
        'Mitigated Steel Infill Size': 0,
        'Mitigated Steel Girder Size': 0,
        'Mitigated Steel EC_Scaled': 0,
        'Mitigated Steel Peak Acceleration': 0,
        'Mitigated Steel Fundamental Frequency': 0,
        'Best Infill Size': 0,
        'Best Girder Size': 0,
        'Best Topping': 0,
        'Best EC_Scaled': 0,
        'Best Peak Acceleration': 0,
        'Best Fundamental Frequency': 0,
        'Best Method': 0, 
    })

else:
    initial_values, infill_values, girder_values, conc_values, steel_values, best_method, condition_class = single_results
    single_data.append({
        'Initial Infill Size': initial_values['Infill Sizes'].iloc[0],
        'Initial Girder Size': initial_values['Girder Sizes'].iloc[0],
        'Initial Topping': conc_values['Total Topping'].iloc[0],
        'Initial EC_Scaled': initial_values['Embodied Carbon Scaled'].iloc[0],
        'Initial Peak Acceleration': initial_values['Peak Acceleration'].iloc[0],
        'Initial Fundamental Frequency': initial_values['Fundamental Frequency'].iloc[0],
        'Mitigated Infill # of Upsizes': infill_values['# of Upsizes'].iloc[-1],
        'Mitigated Infill Size': infill_values['Infill Sizes'].iloc[-1],
        'Mitigated Infill Girder Size': infill_values['Girder Sizes'].iloc[-1],
        'Mitigated Infill EC_Scaled': infill_values['Embodied Carbon Scaled'].iloc[-1],
        'Mitigated Infill Peak Acceleration': infill_values['Peak Acceleration'].iloc[-1],
        'Mitigated Infill Fundamental Frequency': infill_values['Fundamental Frequency'].iloc[-1],
        'Mitigated Girder # of Upsizes': girder_values['# of Upsizes'].iloc[-1],
        'Mitigated Girder Size': girder_values['Girder Sizes'].iloc[-1],
        'Mitigated Girder Infill Size': girder_values['Infill Sizes'].iloc[-1],
        'Mitigated Girder EC_Scaled': girder_values['Embodied Carbon Scaled'].iloc[-1],
        'Mitigated Girder Peak Acceleration': girder_values['Peak Acceleration'].iloc[-1],
        'Mitigated Girder Fundamental Frequency': girder_values['Fundamental Frequency'].iloc[-1],
        'Mitigated Conc Additional Topping (in)': conc_values['Additional Topping'].iloc[-1],
        'Mitigated Conc Infill Size': conc_values['Infill Sizes'].iloc[-1],
        'Mitigated Conc Girder Size': conc_values['Girder Sizes'].iloc[-1],
        'Mitigated Conc Topping': conc_values['Total Topping'].iloc[-1],
        'Mitigated Conc EC_Scaled': conc_values['Embodied Carbon Scaled'].iloc[-1],
        'Mitigated Conc Peak Acceleration': conc_values['Peak Acceleration'].iloc[-1],
        'Mitigated Conc Fundamental Frequency': conc_values['Fundamental Frequency'].iloc[-1],
        'Mitigated Steel Infill Size': steel_values['Infill Sizes'].iloc[0],
        'Mitigated Steel Girder Size': steel_values['Girder Sizes'].iloc[0],
        'Mitigated Steel EC_Scaled': steel_values['Embodied Carbon Scaled'].iloc[0],
        'Mitigated Steel Peak Acceleration': steel_values['Peak Acceleration'].iloc[0],
        'Mitigated Steel Fundamental Frequency': steel_values['Fundamental Frequency'].iloc[0],
        'Best Infill Size': best_method['Infill Sizes'].iloc[-1],
        'Best Girder Size': best_method['Girder Sizes'].iloc[-1],
        'Best Topping': best_method['Total Topping'].iloc[-1],
        'Best EC_Scaled': best_method['Embodied Carbon Scaled'].iloc[-1],
        'Best Peak Acceleration': best_method['Peak Acceleration'].iloc[-1],
        'Best Fundamental Frequency': best_method['Fundamental Frequency'].iloc[-1],
        'Best Method': condition_class, 
    })

# Create a DataFrame from the results list
single_df = pd.DataFrame(single_data)

#Uncomment whichever df to be shown
#initial_values
#infill_values
#girder_values
#conc_values
#best_method
#infill_values
#test_dataframe
single_df
single_df